# 3. 使用 Zeo++ 进行孔隙分析

语言：[English](../en/03_zeopp_pore_analysis.ipynb) | **中文**

本 Notebook 会查找 Notebook 1 生成的 TAPB–TFB CIF，运行 cofkit 的 Zeo++ 分析封装，并输出便于阅读的孔隙信息。CLI 单元格可以直接执行；等价的 `analyze_zeopp_pore_properties` Python 工作流以注释代码形式提供。

## 教程系列

1. [环境与首次构建](01_environment_and_first_build.ipynb)
2. [拓扑、连接数组合与连接键示例](02_topologies_connectivities_and_linkages.ipynb)
3. **使用 Zeo++ 进行孔隙分析**（本 Notebook）

前置条件：成功运行 Notebook 1，并准备好 Zeo++ 0.3 的 `network` 可执行文件。`cofkit` Python 软件包本身无需 Zeo++ 即可工作；只有本分析需要这个外部程序。

## 查找 Notebook 1 的 CIF

构建流程会按粗略验证类别对 CIF 分组，因此具体子目录可能不同。下面的搜索会遍历所有类别；如果 Notebook 1 尚未生成 CIF，则会给出明确的错误信息。

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
CIF_PATH="$(find out/tutorial_01_first_build/cifs -type f -name 'tapb__tfb__hcb.cif' -print -quit 2>/dev/null || true)"
if [[ -z "$CIF_PATH" ]]; then
  echo 'No TAPB–TFB CIF was found. Run Notebook 1 before continuing.' >&2
  exit 1
fi
printf 'Using CIF: %s\n' "$CIF_PATH"
sed -n '1,18p' "$CIF_PATH"

In [ ]:
# Python 等价实现（已注释）：查找 Notebook 1 的 CIF。
# from pathlib import Path
# matches = sorted(Path("out/tutorial_01_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# if not matches:
#     raise FileNotFoundError("Run Notebook 1 before continuing.")
# cif_path = matches[0]
# print(cif_path)

## 配置 Zeo++

如果已经安装 Zeo++，请在启动 Jupyter 前将 `COFKIT_ZEOPP_PATH` 指向其可执行文件：

```bash
export COFKIT_ZEOPP_PATH=/absolute/path/to/zeo++-0.3/network
```

cofkit CLI 也会从最近的 `.env` 文件读取该变量。若没有已配置的可执行文件，下一个可选单元格会在 `$HOME/.local/opt/zeo++-0.3` 下构建上游 Zeo++ 0.3。如果你在其他位置管理 Zeo++，请跳过该单元格。

In [ ]:
%%bash
set -euo pipefail
if [[ -n "${COFKIT_ZEOPP_PATH:-}" && -x "$COFKIT_ZEOPP_PATH" ]]; then
  printf 'Using configured Zeo++: %s\n' "$COFKIT_ZEOPP_PATH"
  exit 0
fi
INSTALL_PARENT="$HOME/.local/opt"
INSTALL_DIR="$INSTALL_PARENT/zeo++-0.3"
ARCHIVE="$INSTALL_PARENT/zeo++-0.3.tar.gz"
mkdir -p "$INSTALL_PARENT"
if [[ ! -x "$INSTALL_DIR/network" ]]; then
  command -v curl >/dev/null || { echo 'curl is required to download Zeo++.' >&2; exit 1; }
  command -v make >/dev/null || { echo 'A C++ build toolchain and make are required.' >&2; exit 1; }
  curl -L 'http://www.zeoplusplus.org/zeo++-0.3.tar.gz' -o "$ARCHIVE"
  tar -xzf "$ARCHIVE" -C "$INSTALL_PARENT"
  make -C "$INSTALL_DIR/voro++/src"
  make -C "$INSTALL_DIR"
fi
test -x "$INSTALL_DIR/network"
printf 'Zeo++ is ready at: %s\n' "$INSTALL_DIR/network"
printf '%s\n' 'Later cells use this default path when COFKIT_ZEOPP_PATH is unset.'

In [ ]:
# 编译 Zeo++ 没有对应的 Python API 实现。
# Zeo++ 是外部 C++ 程序；cofkit 的 Python API 从下方的分析步骤开始。

## 运行点探针和有限尺寸探针分析

基线分析会报告限制孔道的球尺寸，以及点探针可接触表面积和体积。重复使用 `--probe-radius` 会增加考虑可达性的探针扫描；这里示范的半径为 1.20 Å 和 1.86 Å。请根据实际研究所用的吸附质模型调整这些半径。

命令有意省略 `--json`，因此终端会输出简洁易读的报告。同时仍会在磁盘上写入完整的机器可读 `zeopp_report.json`。

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
CIF_PATH="$(find out/tutorial_01_first_build/cifs -type f -name 'tapb__tfb__hcb.cif' -print -quit)"
ZEOPP_BIN="${COFKIT_ZEOPP_PATH:-$HOME/.local/opt/zeo++-0.3/network}"
test -n "$CIF_PATH" || { echo 'Run Notebook 1 first.' >&2; exit 1; }
test -x "$ZEOPP_BIN" || { echo "Zeo++ executable not found: $ZEOPP_BIN" >&2; exit 1; }
conda run -n cofkit cofkit analyze zeopp "$CIF_PATH" \
  --zeopp-path "$ZEOPP_BIN" \
  --output-dir out/tutorial_03_zeopp \
  --probe-radius 1.20 \
  --probe-radius 1.86 \
  --surface-samples-per-atom 250 \
  --volume-samples-total 5000

In [ ]:
# Python API 等价实现（已注释），包括便于阅读的报告。
# import os
# from pathlib import Path
# from cofkit import analyze_zeopp_pore_properties
#
# cif_path = next(Path("out/tutorial_01_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# zeopp_binary = os.environ.get(
#     "COFKIT_ZEOPP_PATH",
#     str(Path.home() / ".local/opt/zeo++-0.3/network"),
# )
# result = analyze_zeopp_pore_properties(
#     cif_path,
#     output_dir="out/tutorial_03_zeopp_python",
#     probe_radii=(1.20, 1.86),
#     surface_samples_per_atom=250,
#     volume_samples_total=5000,
#     zeopp_path=zeopp_binary,
# )
# basic = result.baseline.basic_pore_properties
# channels = result.baseline.point_probe_channels
# surface = result.baseline.point_probe_surface_area
# volume = result.baseline.point_probe_volume
# print(f"Input CIF: {result.input_cif}")
# print(f"Largest included sphere: {basic.largest_included_sphere:.3f} Å")
# print(f"Largest free sphere: {basic.largest_free_sphere:.3f} Å")
# print("Largest included sphere along a free path: "
#       f"{basic.largest_included_sphere_along_free_path:.3f} Å")
# print(f"Point-probe channels / pockets: {channels.n_channels} / {channels.n_pockets}")
# print(f"Point-probe accessible surface area: {surface.accessible_surface_area_a2:.3f} Å²")
# print(f"Point-probe accessible volume: {volume.accessible_volume_a3:.3f} Å³")
# for scan in result.probe_scans:
#     print(f"Probe radius {scan.settings.probe_radius:.2f} Å: {scan.status}")
#     if scan.surface_area is not None and scan.volume is not None:
#         print(f"  accessible surface area = {scan.surface_area.accessible_surface_area_a2:.3f} Å²")
#         print(f"  accessible volume fraction = {scan.volume.accessible_volume_fraction:.5f}")
#     if scan.accessibility is not None:
#         print(f"  accessible Voronoi-node fraction = {scan.accessibility.accessible_fraction}")
# print(f"Full report: {result.report_path}")

## 如何解读主要数值

| 报告字段 | 实际含义 |
|---|---|
| 最大内含球 | 能够放入孔隙网络某处的最大球直径 |
| 最大自由球 | 能够穿过周期性孔隙网络的最大球直径 |
| 自由路径上的最大内含球 | 可贯通路径上遇到的最大孔腔尺寸 |
| 可接触表面积 | 在 Zeo++ 几何模型下，所选探针能够到达的表面 |
| 可接触体积/体积分数 | 探针可到达的孔隙体积，以绝对值或相对晶胞比例表示 |
| 通道与口袋 | 连通的传输路径与孤立的可接触区域 |

报告中的所有球尺寸和探针设置均采用以埃为基础的单位。表面积和体积估计使用蒙特卡洛采样，因此在正式的收敛性研究中应提高采样数量。

## 查看保留的报告和原始输出

cofkit 会保留 Zeo++ 输入/输出文件和日志，使解析后的报告仍可追溯和审查。

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
printf '%s\n' '--- generated Zeo++ files ---'
find out/tutorial_03_zeopp -type f -print | sort
printf '%s\n' '--- zeopp_report.json (first 140 lines) ---'
sed -n '1,140p' out/tutorial_03_zeopp/zeopp_report.json

In [ ]:
# Python 等价实现（已注释）：读取已保存的报告。
# import json
# from pathlib import Path
# report_path = Path("out/tutorial_03_zeopp/zeopp_report.json")
# report = json.loads(report_path.read_text())
# print(report["baseline"]["basic_pore_properties"])
# for scan in report["probe_scans"]:
#     print(scan["settings"]["probe_radius"], scan["status"])

## 使用这些数值之前

孔隙描述符的意义取决于输入几何结构和原子半径模型。本教程生成的 CIF 仅用于演示；在科研计算中，应先优化并验证框架结构，记录 Zeo++ 版本和半径定义，并检查采样收敛性。有限尺寸探针扫描即使失败，也只会记录在 JSON 报告中，不会丢弃已成功获得的基线结果。